# Stage 7B — Deterministic Qualitative Failure Audit

This is **analysis-only inference** at the already-frozen Stage 6 operating point.

It does not train, fine-tune, adapt, select checkpoints, or tune thresholds.

The audit identifies safety-class ground-truth objects missed by **all three source-domain seeds** in each transfer direction and saves deterministic example overlays to Google Drive.

In [ ]:
# Cell 1 — Mount Drive and verify GPU
from pathlib import Path
from google.colab import drive
import shutil, subprocess

drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
RUN_ROOT = DRIVE / 'Stage5_Baselines'
OUT = DRIVE / 'Stage7_Qualitative_Failure_Audit'
OUT.mkdir(parents=True, exist_ok=True)

if not shutil.which('nvidia-smi'):
    raise RuntimeError('GPU runtime required. Choose Runtime > Change runtime type > T4 GPU.')
subprocess.run(['nvidia-smi'], check=True)
print('STAGE7B_GPU_PREFLIGHT: PASS')

In [ ]:
# Cell 2 — Reassemble and verify the same frozen h9-v2 data bundle
import hashlib, zipfile, shutil

PARTS = [
    DRIVE / 'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_00',
    DRIVE / 'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_01',
    DRIVE / 'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_02',
]
missing = [str(p) for p in PARTS if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing frozen bundle parts: ' + ', '.join(missing))

BUNDLE = Path('/content/Stage5_Frozen_h9_v2_Data_Bundle.zip')
EXPECTED_SHA = 'dd8ce20f2650848133ce4323b00139684b0d4ac8b7db618e8df469244221cee9'
h = hashlib.sha256()
with BUNDLE.open('wb') as out:
    for p in PARTS:
        with p.open('rb') as f:
            for chunk in iter(lambda: f.read(8*1024*1024), b''):
                out.write(chunk); h.update(chunk)
assert h.hexdigest() == EXPECTED_SHA

DATA = Path('/content/stage7b_frozen_data')
if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as z:
    assert z.testzip() is None
    z.extractall(DATA)

Y_DATA = DATA / 'h9_v2/Y-PPE-h9-v2/data.yaml'
C_DATA = DATA / 'h9_v2/Construction-PPE-h9-v2/data.yaml'
assert Y_DATA.is_file() and C_DATA.is_file()
print('STAGE7B_FROZEN_DATA: PASS')

In [ ]:
# Cell 3 — Clone the frozen Stage 7 analysis branch and install exact evaluation layer
import subprocess, shutil, sys

REPO = Path('/content/Y-PPE-Detection-RFDETR')
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    'git','clone','--branch','stage7-results-analysis','--single-branch',
    'https://github.com/EZZALDEEN049/Y-PPE-Detection-RFDETR.git', str(REPO)
], check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'ultralytics==8.4.156','PyYAML==6.0.3','Pillow'
], check=True)

import ultralytics, torch
assert ultralytics.__version__ == '8.4.156'
assert torch.cuda.is_available()
print('STAGE7B_CODE_AND_ENV: PASS')

In [ ]:
# Cell 4 — Execute deterministic consensus false-negative audit
import subprocess, sys

SCRIPT = REPO / 'cross_domain_reliability/stage7_results_analysis/qualitative_failure_audit.py'
subprocess.run([
    sys.executable, str(SCRIPT),
    '--stage5-root', str(RUN_ROOT),
    '--y-data', str(Y_DATA),
    '--c-data', str(C_DATA),
    '--out', str(OUT),
    '--device', '0',
    '--batch', '8',
    '--imgsz', '640',
    '--conf', '0.25',
    '--nms-iou', '0.70',
    '--match-iou', '0.50',
    '--examples-per-class', '8',
], check=True)

In [ ]:
# Cell 5 — Show the summary
import json, pandas as pd
from IPython.display import display

summary = json.loads((OUT/'stage7_consensus_fn_summary.json').read_text())
print(json.dumps(summary, indent=2))
display(pd.read_csv(OUT/'stage7_consensus_fn_records.csv').head(40))

print('\nSaved to:', OUT)
print('STAGE7B_COMPLETE')